# 06 Expand Training Data

This notebook expands the project dataset by loading more NFL seasons.

The previous models used data from 2023 through 2025. This notebook uses more seasons to give the model a larger training set and test whether more historical data improves prediction accuracy.

Previous best accuracy: 61.40%

In [1]:
# Import packages
import os
import pandas as pd
import nfl_data_py as nfl

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
# Load more schedule data
seasons = list(range(2018, 2026))

schedules = nfl.import_schedules(seasons)

schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
5049,2018_01_ATL_PHI,2018,REG,1,2018-09-06,Thursday,20:20,ATL,12.0,PHI,...,8.0,00-0026143,00-0029567,Matt Ryan,Nick Foles,Dan Quinn,Doug Pederson,John Hussey,PHI00,Lincoln Financial Field
5050,2018_01_BUF_BAL,2018,REG,1,2018-09-09,Sunday,13:00,BUF,3.0,BAL,...,12.0,00-0033958,00-0026158,Nathan Peterman,Joe Flacco,Sean McDermott,John Harbaugh,Shawn Hochuli,BAL00,M&T Bank Stadium
5051,2018_01_PIT_CLE,2018,REG,1,2018-09-09,Sunday,13:00,PIT,21.0,CLE,...,11.0,00-0022924,00-0028118,Ben Roethlisberger,Tyrod Taylor,Mike Tomlin,Hue Jackson,Shawn Smith,CLE00,FirstEnergy Stadium
5052,2018_01_CIN_IND,2018,REG,1,2018-09-09,Sunday,13:00,CIN,34.0,IND,...,NaN,00-0027973,00-0029668,Andy Dalton,Andrew Luck,Marvin Lewis,Frank Reich,Peter Morelli,IND00,Lucas Oil Stadium
5053,2018_01_TEN_MIA,2018,REG,1,2018-09-09,Sunday,13:00,TEN,20.0,MIA,...,7.0,00-0032268,00-0029701,Marcus Mariota,Ryan Tannehill,Mike Vrabel,Adam Gase,Jerome Boger,MIA00,Hard Rock Stadium


In [3]:
# Check the data
schedules.shape
schedules["season"].value_counts().sort_index()
schedules.columns.tolist()

['game_id',
 'season',
 'game_type',
 'week',
 'gameday',
 'weekday',
 'gametime',
 'away_team',
 'away_score',
 'home_team',
 'home_score',
 'location',
 'result',
 'total',
 'overtime',
 'old_game_id',
 'gsis',
 'nfl_detail_id',
 'pfr',
 'pff',
 'espn',
 'ftn',
 'away_rest',
 'home_rest',
 'away_moneyline',
 'home_moneyline',
 'spread_line',
 'away_spread_odds',
 'home_spread_odds',
 'total_line',
 'under_odds',
 'over_odds',
 'div_game',
 'roof',
 'surface',
 'temp',
 'wind',
 'away_qb_id',
 'home_qb_id',
 'away_qb_name',
 'home_qb_name',
 'away_coach',
 'home_coach',
 'referee',
 'stadium_id',
 'stadium']

In [4]:
# Save the larger raw schedule file
os.makedirs("../data/raw", exist_ok=True)

schedules.to_csv("../data/raw/schedules_2018_2025.csv", index=False)

print("Saved raw schedule data.")
print(schedules.shape)

Saved raw schedule data.
(2227, 46)


In [5]:
# Keep completed games only
completed_games = schedules.dropna(subset=["home_score", "away_score"]).copy()

completed_games.shape

(2227, 46)

In [6]:
# Create the target variable
completed_games["home_team_won"] = (
    completed_games["home_score"] > completed_games["away_score"]
).astype(int)

In [7]:
# Create clean game results dataset
game_results = completed_games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won"
    ]
].copy()

game_results["home_point_diff"] = (
    game_results["home_score"] - game_results["away_score"]
)

game_results.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
5049,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0
5050,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0
5051,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0
5052,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0
5053,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0


In [8]:
# Save expanded game results
os.makedirs("../data/processed", exist_ok=True)

game_results.to_csv("../data/processed/game_results_2018_2025.csv", index=False)

print("Saved expanded game results.")
print(game_results.shape)

Saved expanded game results.
(2227, 10)


In [9]:
# Sort games
games = game_results.copy()

games["gameday"] = pd.to_datetime(games["gameday"])

games = games.sort_values(["season", "week", "gameday"]).reset_index(drop=True)

games.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0


In [10]:
# Create home team rows
home_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_rows = home_rows.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_scored",
        "away_score": "points_allowed"
    }
)

home_rows["is_home"] = 1

In [11]:
# Create away team rows
away_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_rows = away_rows.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_scored",
        "home_score": "points_allowed"
    }
)

away_rows["is_home"] = 0

In [12]:
# Combine team-game rows
team_games = pd.concat([home_rows, away_rows], ignore_index=True)

team_games = team_games.sort_values(
    ["team", "season", "week", "gameday"]
).reset_index(drop=True)

team_games.head(10)

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home
0,2018,1,2018_01_WAS_ARI,2018-09-09,ARI,WAS,6.0,24.0,1
1,2018,2,2018_02_ARI_LA,2018-09-16,ARI,LA,0.0,34.0,0
2,2018,3,2018_03_CHI_ARI,2018-09-23,ARI,CHI,14.0,16.0,1
3,2018,4,2018_04_SEA_ARI,2018-09-30,ARI,SEA,17.0,20.0,1
4,2018,5,2018_05_ARI_SF,2018-10-07,ARI,SF,28.0,18.0,0
5,2018,6,2018_06_ARI_MIN,2018-10-14,ARI,MIN,17.0,27.0,0
6,2018,7,2018_07_DEN_ARI,2018-10-18,ARI,DEN,10.0,45.0,1
7,2018,8,2018_08_SF_ARI,2018-10-28,ARI,SF,18.0,15.0,1
8,2018,10,2018_10_ARI_KC,2018-11-11,ARI,KC,14.0,26.0,0
9,2018,11,2018_11_OAK_ARI,2018-11-18,ARI,OAK,21.0,23.0,1


In [13]:
# Add result columns
team_games["point_diff"] = (
    team_games["points_scored"] - team_games["points_allowed"]
)

team_games["team_won"] = (
    team_games["points_scored"] > team_games["points_allowed"]
).astype(int)

team_games.head()

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home,point_diff,team_won
0,2018,1,2018_01_WAS_ARI,2018-09-09,ARI,WAS,6.0,24.0,1,-18.0,0
1,2018,2,2018_02_ARI_LA,2018-09-16,ARI,LA,0.0,34.0,0,-34.0,0
2,2018,3,2018_03_CHI_ARI,2018-09-23,ARI,CHI,14.0,16.0,1,-2.0,0
3,2018,4,2018_04_SEA_ARI,2018-09-30,ARI,SEA,17.0,20.0,1,-3.0,0
4,2018,5,2018_05_ARI_SF,2018-10-07,ARI,SF,28.0,18.0,0,10.0,1


In [14]:
# Create season-long pre-game stats
team_games["games_played_before"] = (
    team_games.groupby(["team", "season"]).cumcount()
)

team_games["points_scored_before"] = (
    team_games.groupby(["team", "season"])["points_scored"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["points_allowed_before"] = (
    team_games.groupby(["team", "season"])["points_allowed"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["point_diff_before"] = (
    team_games.groupby(["team", "season"])["point_diff"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["wins_before"] = (
    team_games.groupby(["team", "season"])["team_won"]
    .transform(lambda x: x.cumsum().shift(1))
)

cols_to_fill = [
    "points_scored_before",
    "points_allowed_before",
    "point_diff_before",
    "wins_before"
]

team_games[cols_to_fill] = team_games[cols_to_fill].fillna(0)

In [15]:
# Create season-long averages
team_games["avg_points_scored_before"] = 0.0
team_games["avg_points_allowed_before"] = 0.0
team_games["avg_point_diff_before"] = 0.0
team_games["win_pct_before"] = 0.0

mask = team_games["games_played_before"] > 0

team_games.loc[mask, "avg_points_scored_before"] = (
    team_games.loc[mask, "points_scored_before"] /
    team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "avg_points_allowed_before"] = (
    team_games.loc[mask, "points_allowed_before"] /
    team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "avg_point_diff_before"] = (
    team_games.loc[mask, "point_diff_before"] /
    team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "win_pct_before"] = (
    team_games.loc[mask, "wins_before"] /
    team_games.loc[mask, "games_played_before"]
)

In [16]:
# Create last-3-game features
team_games["last3_avg_points_scored"] = (
    team_games.groupby(["team", "season"])["points_scored"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

team_games["last3_avg_points_allowed"] = (
    team_games.groupby(["team", "season"])["points_allowed"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

team_games["last3_avg_point_diff"] = (
    team_games.groupby(["team", "season"])["point_diff"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

team_games["last3_win_pct"] = (
    team_games.groupby(["team", "season"])["team_won"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

last3_cols = [
    "last3_avg_points_scored",
    "last3_avg_points_allowed",
    "last3_avg_point_diff",
    "last3_win_pct"
]

team_games[last3_cols] = team_games[last3_cols].fillna(0)

In [17]:
# Check one team
team_games[team_games["team"] == "KC"][
    [
        "season",
        "week",
        "team",
        "opponent",
        "points_scored",
        "points_allowed",
        "games_played_before",
        "avg_points_scored_before",
        "last3_avg_points_scored",
        "avg_point_diff_before",
        "last3_avg_point_diff",
        "win_pct_before",
        "last3_win_pct"
    ]
].head(20)

,season,week,team,opponent,points_scored,points_allowed,games_played_before,avg_points_scored_before,last3_avg_points_scored,avg_point_diff_before,last3_avg_point_diff,win_pct_before,last3_win_pct
2070,2018,1,KC,LAC,38.0,28.0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2071,2018,2,KC,PIT,42.0,37.0,1,38.000000,38.000000,10.000000,10.000000,1.000000,1.000000
2072,2018,3,KC,SF,38.0,27.0,2,40.000000,40.000000,7.500000,7.500000,1.000000,1.000000
2073,2018,4,KC,DEN,27.0,23.0,3,39.333333,39.333333,8.666667,8.666667,1.000000,1.000000
2074,2018,5,KC,JAX,30.0,14.0,4,36.250000,35.666667,7.500000,6.666667,1.000000,1.000000
2075,2018,6,KC,NE,40.0,43.0,5,35.000000,31.666667,9.200000,10.333333,1.000000,1.000000
2076,2018,7,KC,CIN,45.0,10.0,6,35.833333,32.333333,7.166667,5.666667,0.833333,0.666667
2077,2018,8,KC,DEN,30.0,23.0,7,37.142857,38.333333,11.142857,16.000000,0.857143,0.666667
2078,2018,9,KC,CLE,37.0,21.0,8,36.250000,38.333333,10.625000,13.000000,0.875000,0.666667
2079,2018,10,KC,ARI,26.0,14.0,9,36.333333,37.333333,11.222222,19.333333,0.888889,1.000000


In [18]:
# Create home features
feature_cols = [
    "game_id",
    "team",
    "games_played_before",
    "avg_points_scored_before",
    "avg_points_allowed_before",
    "avg_point_diff_before",
    "win_pct_before",
    "last3_avg_points_scored",
    "last3_avg_points_allowed",
    "last3_avg_point_diff",
    "last3_win_pct"
]

home_features = team_games[team_games["is_home"] == 1][feature_cols].copy()

home_features = home_features.rename(
    columns={
        "team": "home_team_check",
        "games_played_before": "home_games_played_before",
        "avg_points_scored_before": "home_avg_points_scored_before",
        "avg_points_allowed_before": "home_avg_points_allowed_before",
        "avg_point_diff_before": "home_avg_point_diff_before",
        "win_pct_before": "home_win_pct_before",
        "last3_avg_points_scored": "home_last3_avg_points_scored",
        "last3_avg_points_allowed": "home_last3_avg_points_allowed",
        "last3_avg_point_diff": "home_last3_avg_point_diff",
        "last3_win_pct": "home_last3_win_pct"
    }
)

home_features.head()

,game_id,home_team_check,home_games_played_before,home_avg_points_scored_before,home_avg_points_allowed_before,home_avg_point_diff_before,home_win_pct_before,home_last3_avg_points_scored,home_last3_avg_points_allowed,home_last3_avg_point_diff,home_last3_win_pct
0,2018_01_WAS_ARI,ARI,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2018_03_CHI_ARI,ARI,2,3.000000,29.000000,-26.000000,0.000000,3.000000,29.000000,-26.000000,0.000000
3,2018_04_SEA_ARI,ARI,3,6.666667,24.666667,-18.000000,0.000000,6.666667,24.666667,-18.000000,0.000000
6,2018_07_DEN_ARI,ARI,6,13.666667,23.166667,-9.500000,0.166667,20.666667,21.666667,-1.000000,0.333333
7,2018_08_SF_ARI,ARI,7,13.142857,26.285714,-13.142857,0.142857,18.333333,30.000000,-11.666667,0.333333


In [19]:
# Create away features
away_features = team_games[team_games["is_home"] == 0][feature_cols].copy()

away_features = away_features.rename(
    columns={
        "team": "away_team_check",
        "games_played_before": "away_games_played_before",
        "avg_points_scored_before": "away_avg_points_scored_before",
        "avg_points_allowed_before": "away_avg_points_allowed_before",
        "avg_point_diff_before": "away_avg_point_diff_before",
        "win_pct_before": "away_win_pct_before",
        "last3_avg_points_scored": "away_last3_avg_points_scored",
        "last3_avg_points_allowed": "away_last3_avg_points_allowed",
        "last3_avg_point_diff": "away_last3_avg_point_diff",
        "last3_win_pct": "away_last3_win_pct"
    }
)

away_features.head()

,game_id,away_team_check,away_games_played_before,away_avg_points_scored_before,away_avg_points_allowed_before,away_avg_point_diff_before,away_win_pct_before,away_last3_avg_points_scored,away_last3_avg_points_allowed,away_last3_avg_point_diff,away_last3_win_pct
1,2018_02_ARI_LA,ARI,1,6.00,24.000,-18.000,0.00,6.000000,24.000000,-18.000000,0.000000
4,2018_05_ARI_SF,ARI,4,9.25,23.500,-14.250,0.00,10.333333,23.333333,-13.000000,0.000000
5,2018_06_ARI_MIN,ARI,5,13.00,22.400,-9.400,0.20,19.666667,18.000000,1.666667,0.333333
8,2018_10_ARI_KC,ARI,8,13.75,24.875,-11.125,0.25,15.000000,29.000000,-14.000000,0.333333
10,2018_12_ARI_LAC,ARI,10,14.50,24.800,-10.300,0.20,17.666667,21.333333,-3.666667,0.333333


In [ ]:
# Merge features onto games
modeling_data_expanded = games.merge(home_features, on="game_id", how="left")

modeling_data_expanded = modeling_data_expanded.merge(
    away_features,
    on="game_id",
    how="left"
)

modeling_data_expanded.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,away_team_check,away_games_played_before,away_avg_points_scored_before,away_avg_points_allowed_before,away_avg_point_diff_before,away_win_pct_before,away_last3_avg_points_scored,away_last3_avg_points_allowed,away_last3_avg_point_diff,away_last3_win_pct
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,ATL,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,BUF,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,PIT,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,CIN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,TEN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
# Check team matches
modeling_data_expanded[
    [
        "game_id",
        "home_team",
        "home_team_check",
        "away_team",
        "away_team_check"
    ]
].head(10)

,game_id,home_team,home_team_check,away_team,away_team_check
0,2018_01_ATL_PHI,PHI,PHI,ATL,ATL
1,2018_01_BUF_BAL,BAL,BAL,BUF,BUF
2,2018_01_PIT_CLE,CLE,CLE,PIT,PIT
3,2018_01_CIN_IND,IND,IND,CIN,CIN
4,2018_01_TEN_MIA,MIA,MIA,TEN,TEN
5,2018_01_SF_MIN,MIN,MIN,SF,SF
6,2018_01_HOU_NE,NE,NE,HOU,HOU
7,2018_01_TB_NO,NO,NO,TB,TB
8,2018_01_JAX_NYG,NYG,NYG,JAX,JAX
9,2018_01_KC_LAC,LAC,LAC,KC,KC


In [22]:
# Drop check columns
modeling_data_expanded = modeling_data_expanded.drop(
    columns=["home_team_check", "away_team_check"]
)

In [23]:
# Create difference features
modeling_data_expanded["avg_points_scored_diff"] = (
    modeling_data_expanded["home_avg_points_scored_before"] -
    modeling_data_expanded["away_avg_points_scored_before"]
)

modeling_data_expanded["avg_points_allowed_diff"] = (
    modeling_data_expanded["home_avg_points_allowed_before"] -
    modeling_data_expanded["away_avg_points_allowed_before"]
)

modeling_data_expanded["avg_point_diff_diff"] = (
    modeling_data_expanded["home_avg_point_diff_before"] -
    modeling_data_expanded["away_avg_point_diff_before"]
)

modeling_data_expanded["win_pct_diff"] = (
    modeling_data_expanded["home_win_pct_before"] -
    modeling_data_expanded["away_win_pct_before"]
)

modeling_data_expanded["last3_avg_points_scored_diff"] = (
    modeling_data_expanded["home_last3_avg_points_scored"] -
    modeling_data_expanded["away_last3_avg_points_scored"]
)

modeling_data_expanded["last3_avg_points_allowed_diff"] = (
    modeling_data_expanded["home_last3_avg_points_allowed"] -
    modeling_data_expanded["away_last3_avg_points_allowed"]
)

modeling_data_expanded["last3_avg_point_diff_diff"] = (
    modeling_data_expanded["home_last3_avg_point_diff"] -
    modeling_data_expanded["away_last3_avg_point_diff"]
)

modeling_data_expanded["last3_win_pct_diff"] = (
    modeling_data_expanded["home_last3_win_pct"] -
    modeling_data_expanded["away_last3_win_pct"]
)

In [24]:
# Save expanded modeling dataset
modeling_data_expanded.to_csv(
    "../data/processed/modeling_dataset_expanded_2018_2025.csv",
    index=False
)

print("Saved expanded modeling dataset.")
print(modeling_data_expanded.shape)

Saved expanded modeling dataset.
(2227, 36)


In [25]:
# Choose features
features = [
    "avg_points_scored_diff",
    "avg_points_allowed_diff",
    "avg_point_diff_diff",
    "win_pct_diff",
    "last3_avg_points_scored_diff",
    "last3_avg_points_allowed_diff",
    "last3_avg_point_diff_diff",
    "last3_win_pct_diff"
]

target = "home_team_won"

In [26]:
# Train on 2018–2024, test on 2025
train_data = modeling_data_expanded[
    modeling_data_expanded["season"].between(2018, 2024)
].copy()

test_data = modeling_data_expanded[
    modeling_data_expanded["season"] == 2025
].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 1942
Testing rows: 285


In [27]:
# Train logistic regressing model
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [28]:
# Predict 2025 games
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

In [29]:
# Check accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Expanded training data accuracy: {accuracy:.2%}")
print("Previous best accuracy: 61.40%")

Expanded training data accuracy: 62.46%
Previous best accuracy: 61.40%


In [30]:
# Home-team baseline
home_team_baseline = [1] * len(y_test)

baseline_accuracy = accuracy_score(y_test, home_team_baseline)

print(f"Home-team baseline accuracy: {baseline_accuracy:.2%}")

Home-team baseline accuracy: 53.33%


In [31]:
# Classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.46      0.53       133
           1       0.62      0.77      0.69       152

    accuracy                           0.62       285
   macro avg       0.63      0.61      0.61       285
weighted avg       0.63      0.62      0.61       285



In [32]:
# Model coefficients
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

coefficients.sort_values("coefficient", ascending=False)

,feature,coefficient
7,last3_win_pct_diff,0.439613
3,win_pct_diff,0.049852
0,avg_points_scored_diff,0.029989
2,avg_point_diff_diff,0.025656
1,avg_points_allowed_diff,0.004333
5,last3_avg_points_allowed_diff,0.004112
4,last3_avg_points_scored_diff,0.001387
6,last3_avg_point_diff_diff,-0.002725


In [33]:
# Create results table
results = test_data[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won"
    ]
].copy()

results["predicted_home_win"] = y_pred
results["home_win_probability"] = y_prob

results["predicted_winner"] = results.apply(
    lambda row: row["home_team"] if row["predicted_home_win"] == 1 else row["away_team"],
    axis=1
)

results["actual_winner"] = results.apply(
    lambda row: row["home_team"] if row["home_team_won"] == 1 else row["away_team"],
    axis=1
)

results["correct_prediction"] = (
    results["predicted_winner"] == results["actual_winner"]
)

results.head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,predicted_home_win,home_win_probability,predicted_winner,actual_winner,correct_prediction
1942,2025,1,2025_01_DAL_PHI,2025-09-04,PHI,DAL,24.0,20.0,1,1,0.547025,PHI,PHI,True
1943,2025,1,2025_01_KC_LAC,2025-09-05,LAC,KC,27.0,21.0,1,1,0.547025,LAC,LAC,True
1944,2025,1,2025_01_TB_ATL,2025-09-07,ATL,TB,20.0,23.0,0,1,0.547025,ATL,TB,False
1945,2025,1,2025_01_CIN_CLE,2025-09-07,CLE,CIN,16.0,17.0,0,1,0.547025,CLE,CIN,False
1946,2025,1,2025_01_MIA_IND,2025-09-07,IND,MIA,33.0,8.0,1,1,0.547025,IND,IND,True
1947,2025,1,2025_01_CAR_JAX,2025-09-07,JAX,CAR,26.0,10.0,1,1,0.547025,JAX,JAX,True
1948,2025,1,2025_01_LV_NE,2025-09-07,NE,LV,13.0,20.0,0,1,0.547025,NE,LV,False
1949,2025,1,2025_01_ARI_NO,2025-09-07,NO,ARI,13.0,20.0,0,1,0.547025,NO,ARI,False
1950,2025,1,2025_01_PIT_NYJ,2025-09-07,NYJ,PIT,32.0,34.0,0,1,0.547025,NYJ,PIT,False
1951,2025,1,2025_01_NYG_WAS,2025-09-07,WAS,NYG,21.0,6.0,1,1,0.547025,WAS,WAS,True


In [34]:
# Accuracy by week
accuracy_by_week = (
    results.groupby("week")["correct_prediction"]
    .mean()
    .reset_index()
)

accuracy_by_week["accuracy_percent"] = (
    accuracy_by_week["correct_prediction"] * 100
)

accuracy_by_week = accuracy_by_week.drop(columns=["correct_prediction"])

accuracy_by_week

,week,accuracy_percent
0,1,56.250000
1,2,75.000000
2,3,56.250000
3,4,50.000000
4,5,42.857143
5,6,46.666667
6,7,73.333333
7,8,61.538462
8,9,57.142857
9,10,71.428571


In [35]:
# Accuracy by confidence level
def get_confidence(prob):
    if prob >= 0.65 or prob <= 0.35:
        return "High"
    elif prob >= 0.57 or prob <= 0.43:
        return "Medium"
    else:
        return "Low"

results["confidence"] = results["home_win_probability"].apply(get_confidence)

confidence_accuracy = (
    results.groupby("confidence")["correct_prediction"]
    .agg(["count", "mean"])
    .reset_index()
)

confidence_accuracy["accuracy_percent"] = confidence_accuracy["mean"] * 100

confidence_accuracy = confidence_accuracy.rename(
    columns={"count": "number_of_games"}
)

confidence_accuracy = confidence_accuracy.drop(columns=["mean"])

confidence_accuracy

,confidence,number_of_games,accuracy_percent
0,High,81,72.839506
1,Low,123,58.536585
2,Medium,81,58.024691


In [36]:
# Save expanded model predictions
results.to_csv(
    "../data/predictions/expanded_training_model_test_predictions.csv",
    index=False
)

print("Saved expanded model test predictions.")
print(results.shape)

Saved expanded model test predictions.
(285, 15)


In [37]:
# Final summary numbers
print("Expanded training data accuracy:", f"{accuracy:.2%}")
print("Previous best accuracy: 61.40%")
print("Home-team baseline accuracy:", f"{baseline_accuracy:.2%}")
print("Number of test games:", len(results))
print("Correct predictions:", results["correct_prediction"].sum())
print("Incorrect predictions:", len(results) - results["correct_prediction"].sum())

Expanded training data accuracy: 62.46%
Previous best accuracy: 61.40%
Home-team baseline accuracy: 53.33%
Number of test games: 285
Correct predictions: 178
Incorrect predictions: 107


## Summary

In this notebook, I expanded the training data from 2023–2025 to 2018–2025.

The model trained on games from 2018 through 2024 and tested on the 2025 season. This gives the model more historical examples to learn from and creates a more stable forecasting test.

The model still uses basic season-long and last-3-game features. The next major improvement should be adding stronger football-specific features, such as EPA/play, offensive efficiency, defensive efficiency, Elo ratings, or rest-day information.